# Phase 3 — Data Preprocessing

## Goal

Clean, deduplicated, stratified, and tokenized HumAID data that will be used consistently by both the baseline model and the LLM.

The purpose of this phase is to ensure that later model comparisons are apples-to-apples.

### Planned steps

1. Clean tweet text and handle Unicode/retweet noise
2. Remove duplicate tweets
3. Create stratified train/validation/test splits
4. Inspect Qwen tokenizer behavior
5. Analyze token-length distribution
6. Save processed splits
7. Move reusable preprocessing logic into `src/data_utils.py`

In [ ]:
import re

def clean_tweet(text):
    # Remove/normalize invalid UTF-8 sequences
    text = text.encode("utf-8", "ignore").decode("utf-8")
    
    # Remove RT @username: prefix
    text = re.sub(r"^RT\s+@\w+:\s*", "", text)
    
    return text.strip()

In [ ]:
examples = [
    "RT @someone: This is a disaster update",
    "California wildfires are getting worse 😢",
    "People need #water and #food urgently"
]

for text in examples:
    print("Original:", text)
    print("Cleaned :", clean_tweet(text))
    print()

In [ ]:
from datasets import load_dataset

ds = load_dataset("QCRI/HumAID-all", split="train")

df = ds.to_pandas()

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
df["text_clean"] = df["tweet_text"].apply(clean_tweet)

print(df[["tweet_text", "text_clean"]].head())

## Step 2 — Deduplication

Duplicate tweets are removed after cleaning.

This is important because repeated or near-identical tweets can otherwise appear in different splits and cause data leakage, making model performance look artificially high.

Deduplication is performed using the cleaned tweet text.

In [ ]:
print("Before deduplication:", len(df))

df = df.drop_duplicates(
    subset="text_clean"
).reset_index(drop=True)

print("After deduplication:", len(df))
print("Duplicates removed:", 53531 - len(df))

In [ ]:
print(df.shape)
print(df.columns.tolist())

## Step 3 — stratified train/validation/test split.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["class_label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["class_label"],
    random_state=42
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

In [ ]:
for name, split in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df)
]:
    print(f"\n{name}:")
    print(split["class_label"].value_counts(normalize=True).round(4))

### Split Verification

The dataset was split using stratified sampling with a fixed random seed of 42.

The class proportions remain nearly identical across the train, validation, and test sets. The rare `missing_or_found_people` class is also present in all three splits, representing approximately 0.46–0.47% of each split.

This confirms that the stratified split preserved the original class distribution and avoided losing the rarest class from any split.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct"
)

sample = train_df["text_clean"].iloc[0]

tokens = tokenizer.tokenize(sample)

print("Tweet:")
print(sample)

print("\nTokens:")
print(tokens)

print("\nNumber of tokens:", len(tokens))

In [ ]:
samples = train_df["text_clean"].sample(5, random_state=42)

for i, text in enumerate(samples, 1):
    tokens = tokenizer.tokenize(text)

    print("=" * 80)
    print(f"Sample {i}")
    print("=" * 80)

    print("Tweet:")
    print(text)

    print("\nTokens:")
    print(tokens)

    print("\nToken count:", len(tokens))

In [ ]:
train_df["token_length"] = train_df["text_clean"].apply(
    lambda x: len(tokenizer.tokenize(x))
)

print(train_df["token_length"].describe())

In [ ]:
print("50th percentile:", train_df["token_length"].quantile(0.50))
print("90th percentile:", train_df["token_length"].quantile(0.90))
print("95th percentile:", train_df["token_length"].quantile(0.95))
print("99th percentile:", train_df["token_length"].quantile(0.99))
print("Maximum:", train_df["token_length"].max())

In [ ]:
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(processed_dir / "train.csv", index=False)
val_df.to_csv(processed_dir / "val.csv", index=False)
test_df.to_csv(processed_dir / "test.csv", index=False)

print("Saved processed datasets:")
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))


In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [ ]:
from src.data_utils import clean_tweet

test_text = "RT @someone: Please help the people affected by the earthquake."

print("Original:")
print(test_text)

print("\nCleaned:")
print(clean_tweet(test_text))

## Preprocessing Decision: RT Prefix

Retweet prefixes such as `RT @username:` are removed during preprocessing.

The reason is to reduce metadata and user-specific noise while keeping the actual humanitarian content of the tweet. The retweet marker was not considered necessary semantic information for the 11-class humanitarian classification task.

## Deduplication Result

Before deduplication, the dataset contained 53,531 tweets.

After removing duplicate cleaned tweets, 53,530 tweets remained.

Only 1 duplicate was removed, so duplicate leakage does not appear to be a major issue in this dataset.